In [ ]:
import pandas as pd
import pandas as pd
import numpy as np
from sklearn.metrics import precision_recall_fscore_support
from statsmodels.stats import inter_rater as irr
from sklearn.metrics import f1_score
from functools import reduce

In [ ]:
# load df with the results of experiments for all model-prompt configurations

In [ ]:

suffixes = [
    "gpt4_baseline",
    "gpt4_ext_ctx",  # extended definitions + Russia specific
    "gpt4_ext",  # extended definitions
    "gpt5_baseline",  #old baseline prompt
    "gemini_baseline",
    "gpt5_ext_ctx",  # extended definitions + Russia specific
    "gemini_ext_ctx",
    "gpt5_ext",  # extended definitions
    "gemini_ext",  
    "gpt4_ext3",
    "gpt5_ext3",
    "gemini_ext3", #bias-calibrated
    "gemini_ext3_v2",
    "gpt4_ext3RU",
    "gpt5_ext3RU",
    "gemini_ext3RU" 
]

k = 3

for val in values_list:
    
    col_e = f"{val}_e"
    df[col_e] = (
        df[f"{val}_e_raw"]
        .astype(str)
        .str.split(',')
        .apply(lambda xs: int(sum(1 for x in xs if x.strip() == '1') >= 2)) 
    )
    
    # --- LLM with multiple raw votes ---
    for suffix in suffixes:
        col = f"{val}_{suffix}"
        raw_col = f"{val}_{suffix}_raw"
        df[col] = (
            df[raw_col]
            .astype(str)
            .fillna("")
            .str.split(",")
            .apply(lambda xs: int(sum(1 for x in xs if x.strip() == "1") >= k))
        )



Robustness check

In [56]:
def count_ones(s):
    if pd.isna(s):
        return np.nan
    return sum(int(x.strip()) for x in str(s).split(","))

def robustness_analysis(df, values, k=3):

    results = []

    for v in values:
        col_v1 = f"{v}_gemini_ext3_raw"
        col_v2 = f"{v}_gemini_ext3_v2_raw"

        # vote counts
        df[f"{v}_v1_count"] = df[col_v1].apply(count_ones)
        df[f"{v}_v2_count"] = df[col_v2].apply(count_ones)

        # majority decisions
        df[f"{v}_v1_majority"] = (df[f"{v}_v1_count"] >= k).astype(int)
        df[f"{v}_v2_majority"] = (df[f"{v}_v2_count"] >= k).astype(int)

        # --- I. Majority agreement ---
        majority_agreement = (
            df[f"{v}_v1_majority"] == df[f"{v}_v2_majority"]
        ).mean()

        # Cohen's kappa
        from sklearn.metrics import cohen_kappa_score
        kappa = cohen_kappa_score(
            df[f"{v}_v1_majority"],
            df[f"{v}_v2_majority"]
        )

        # --- II. Vote count volatility ---
        vote_corr = df[[f"{v}_v1_count", f"{v}_v2_count"]].corr().iloc[0,1]

        mean_abs_diff = (
            df[f"{v}_v1_count"] - df[f"{v}_v2_count"]
        ).abs().mean()

        std_diff = (
            df[f"{v}_v1_count"] - df[f"{v}_v2_count"]
        ).std()

        results.append({
            "Value": v,
            "Majority agreement": round(majority_agreement, 3),
            "Cohen kappa": round(kappa, 3),
            "Vote count correlation": round(vote_corr, 3),
            "Mean |Δ votes|": round(mean_abs_diff, 3),
            "Std Δ votes": round(std_diff, 3)
        })

    return pd.DataFrame(results)

In [57]:
dfr=robustness_analysis(df, values_list, k=3)
dfr

,Value,Majority agreement,Cohen kappa,Vote count correlation,Mean |Δ votes|,Std Δ votes
0,Self-direction,0.963,0.852,0.928,0.225,0.604
1,Stimulation,0.978,0.841,0.904,0.171,0.535
2,Hedonism,0.965,0.842,0.906,0.241,0.635
3,Achievement,0.975,0.886,0.953,0.137,0.470
4,Power,0.982,0.824,0.922,0.111,0.411
5,Security,0.945,0.807,0.906,0.327,0.714
6,Conformity,0.980,0.633,0.807,0.125,0.449
7,Tradition,0.975,0.862,0.932,0.158,0.509
8,Benevolence,0.956,0.912,0.961,0.235,0.654
9,Universalism,0.979,0.899,0.939,0.171,0.503


In [58]:
all_diffs = []

for v in values_list:
    diff = abs(df[f"{v}_v1_count"] - df[f"{v}_v2_count"])
    all_diffs.extend(diff)

np.mean(all_diffs)

np.float64(0.1901)

In [59]:
def overall_majority_agreement(df, values, k=3):

    all_v1 = []
    all_v2 = []

    for v in values:
        v1 = (df[f"{v}_v1_count"] >= k).astype(int)
        v2 = (df[f"{v}_v2_count"] >= k).astype(int)
        all_v1.extend(v1)
        all_v2.extend(v2)

    overall_agreement = np.mean(np.array(all_v1) == np.array(all_v2))

    from sklearn.metrics import cohen_kappa_score
    overall_kappa = cohen_kappa_score(all_v1, all_v2)

    return {
        "Overall majority agreement": round(overall_agreement, 3),
        "Overall Cohen kappa": round(overall_kappa, 3)
    }

In [60]:
overall_majority_agreement(df, values_list, k=3)

{'Overall majority agreement': np.float64(0.97),
 'Overall Cohen kappa': np.float64(0.875)}

Per-expert agreement

In [61]:
metrics = []

for val in values_list:
    y_true = df[f"{val}_e"]

    row = {"value": val}

    for suffix in suffixes:
        y_pred = df[f"{val}_{suffix}"]

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            average="binary",
            zero_division=0
        )

        row[f"{suffix}: precision"] = round(precision, 3)
        row[f"{suffix}: recall"] = round(recall, 3)
        row[f"{suffix}: f1"] = round(f1, 3)

    metrics.append(row)

metrics_df = pd.DataFrame(metrics)

metrics_df


,value,gpt4_baseline: precision,gpt4_baseline: recall,gpt4_baseline: f1,gpt4_ext_ctx: precision,gpt4_ext_ctx: recall,gpt4_ext_ctx: f1,gpt4_ext: precision,gpt4_ext: recall,gpt4_ext: f1,...,gemini_ext3_v2: f1,gpt4_ext3RU: precision,gpt4_ext3RU: recall,gpt4_ext3RU: f1,gpt5_ext3RU: precision,gpt5_ext3RU: recall,gpt5_ext3RU: f1,gemini_ext3RU: precision,gemini_ext3RU: recall,gemini_ext3RU: f1
0,Self-direction,0.286,0.838,0.427,0.335,0.829,0.477,0.322,0.766,0.453,...,0.605,0.318,0.865,0.465,0.587,0.640,0.612,0.519,0.739,0.610
1,Stimulation,0.261,0.671,0.376,0.380,0.500,0.432,0.337,0.400,0.366,...,0.553,0.407,0.529,0.460,0.492,0.443,0.466,0.538,0.600,0.568
2,Hedonism,0.541,0.254,0.345,0.645,0.330,0.437,0.605,0.234,0.338,...,0.576,0.615,0.550,0.581,0.777,0.416,0.542,0.744,0.569,0.645
3,Achievement,0.664,0.654,0.659,0.699,0.567,0.626,0.706,0.567,0.629,...,0.737,0.646,0.646,0.646,0.733,0.669,0.700,0.692,0.780,0.733
4,Power,0.271,0.471,0.344,0.327,0.471,0.386,0.306,0.441,0.361,...,0.512,0.319,0.441,0.370,0.553,0.618,0.583,0.403,0.794,0.535
5,Security,0.780,0.337,0.471,0.707,0.345,0.464,0.703,0.385,0.497,...,0.582,0.836,0.365,0.508,0.811,0.595,0.686,0.696,0.619,0.655
6,Conformity,0.320,0.436,0.369,0.278,0.491,0.355,0.240,0.418,0.305,...,0.265,0.087,0.182,0.118,0.412,0.255,0.315,0.409,0.327,0.364
7,Tradition,0.356,0.508,0.419,0.465,0.656,0.544,0.382,0.557,0.453,...,0.611,0.109,0.098,0.103,0.491,0.885,0.632,0.378,0.836,0.520
8,Benevolence,0.931,0.614,0.740,0.912,0.726,0.808,0.908,0.673,0.773,...,0.871,0.915,0.779,0.842,0.905,0.817,0.859,0.908,0.844,0.875
9,Universalism,0.375,0.434,0.402,0.371,0.398,0.384,0.408,0.349,0.377,...,0.637,0.346,0.434,0.385,0.605,0.554,0.579,0.575,0.735,0.646


In [62]:
# Main expert benchmark

metrics_croped=metrics_df[['value', 'gemini_ext3: precision', 'gemini_ext3: recall', 'gemini_ext3: f1']] # 'gemini_nr3RU: precision', 'gemini_nr3RU: recall', 'gemini_nr3RU: f1' 'gemini_old: precision', 'gemini_old: recall', 'gemini_old: f1', 
metrics_croped.to_csv(res+"p_r_f1_values_gemini.csv", sep="|", encoding="utf-8")
metrics_croped

,value,gemini_ext3: precision,gemini_ext3: recall,gemini_ext3: f1
0,Self-direction,0.534,0.703,0.607
1,Stimulation,0.481,0.543,0.510
2,Hedonism,0.773,0.488,0.598
3,Achievement,0.724,0.701,0.712
4,Power,0.429,0.706,0.533
5,Security,0.708,0.500,0.586
6,Conformity,0.571,0.291,0.386
7,Tradition,0.547,0.852,0.667
8,Benevolence,0.933,0.827,0.877
9,Universalism,0.573,0.807,0.670


In [63]:
metrics_croped=metrics_df[['value', 'gpt4_ext3: precision', 'gpt4_ext3: recall', 'gpt4_ext3: f1', 'gpt5_ext3: precision', 'gpt5_ext3: recall', 'gpt5_ext3: f1', 'gemini_ext3: precision', 'gemini_ext3: recall', 'gemini_ext3: f1']] #'gemini_old: precision', 'gemini_old: recall', 'gemini_old: f1', 
# metrics_croped.to_csv(res+"p_r_f1_values_gemini.csv", sep="|", encoding="utf-8")
metrics_croped

,value,gpt4_ext3: precision,gpt4_ext3: recall,gpt4_ext3: f1,gpt5_ext3: precision,gpt5_ext3: recall,gpt5_ext3: f1,gemini_ext3: precision,gemini_ext3: recall,gemini_ext3: f1
0,Self-direction,0.324,0.793,0.460,0.625,0.495,0.553,0.534,0.703,0.607
1,Stimulation,0.320,0.571,0.410,0.500,0.371,0.426,0.481,0.543,0.510
2,Hedonism,0.635,0.225,0.332,0.772,0.292,0.424,0.773,0.488,0.598
3,Achievement,0.704,0.543,0.613,0.763,0.559,0.645,0.724,0.701,0.712
4,Power,0.283,0.441,0.345,0.542,0.382,0.448,0.429,0.706,0.533
5,Security,0.820,0.325,0.466,0.760,0.516,0.615,0.708,0.500,0.586
6,Conformity,0.339,0.364,0.351,0.438,0.255,0.322,0.571,0.291,0.386
7,Tradition,0.371,0.590,0.456,0.521,0.820,0.637,0.547,0.852,0.667
8,Benevolence,0.912,0.749,0.823,0.906,0.804,0.852,0.933,0.827,0.877
9,Universalism,0.370,0.410,0.389,0.661,0.470,0.549,0.573,0.807,0.670


In [64]:

global_prf_df = collect_global_prf(
    df=df,
    values_list=values_list,
    suffixes=suffixes,
)

# global_prf_df.to_csv(res+"global_f1.csv", sep="|", encoding="utf-8")
global_prf_df


,precision,recall,f1
model,,,
gpt4_baseline,0.527,0.518,0.522
gpt4_ext_ctx,0.592,0.558,0.575
gpt4_ext,0.577,0.514,0.544
gpt5_baseline,0.628,0.674,0.650
gemini_baseline,0.644,0.681,0.662
gpt5_ext_ctx,0.699,0.627,0.661
gemini_ext_ctx,0.680,0.692,0.686
gpt5_ext,0.705,0.596,0.646
gemini_ext,0.684,0.678,0.681


# Error analysis

Set target value for omission / over-attribution reports

In [66]:
suffix = "gemini_ext3" 

In [67]:
import numpy as np
import pandas as pd

def fp_mask(df, value, suffix):
    """False positives for a given value: expert=0, model=1."""
    return (df[f"{value}_e"] == 0) & (df[f"{value}_{suffix}"] == 1)

def fn_mask(df, value, suffix):
    return (df[f"{value}_e"] == 1) & (df[f"{value}_{suffix}"] == 0)

def fp_rate(df, value, suffix):
    """FP among expert negatives (same meaning as extra_rate_cond)."""
    expert_neg = (df[f"{value}_e"] == 0)
    denom = expert_neg.sum()
    if denom == 0:
        return np.nan
    return float(((df[f"{value}_{suffix}"] == 1) & expert_neg).sum() / denom)

def fn_rate(df, value, suffix):
    """FN among expert positives = omission rate (FN / expert positives)"""
    expert_pos = (df[f"{value}_e"] == 1)
    denom = expert_pos.sum()
    if denom == 0:
        return np.nan
    return float(((df[f"{value}_{suffix}"] == 0) & expert_pos).sum() / denom)


def fp_trigger_by_true_values(df, target, suffix, values_list, min_support=30):
    """
    For FP(target), compute which expert-labeled values co-occur in those posts.

    Returns a table with:
      - P(FP_target | true_u=1)
      - P(FP_target | true_u=0)
      - delta (lift-style difference)
      - support_true_u (how many posts have true_u=1)
    """
    fp = fp_mask(df, target, suffix).values
    out = []

    for u in values_list:
        true_u = (df[f"{u}_e"] == 1).values
        n1 = true_u.sum()
        n0 = (~true_u).sum()

        if n1 < min_support:  # avoid unstable estimates
            continue

        p1 = fp[true_u].mean() if n1 > 0 else np.nan
        p0 = fp[~true_u].mean() if n0 > 0 else np.nan

        out.append({
            "true_value": u,
            "P(FP_target | true=1)": float(p1),
            "P(FP_target | true=0)": float(p0),
            "delta": float(p1 - p0),
            "support_true=1": int(n1),
        })

    res = pd.DataFrame(out).sort_values("delta", ascending=False).reset_index(drop=True)
    return res

def fp_expert_composition(
    df,
    target,
    suffix,
    values_list,
    min_fp=20
):
    """
    P(true_u = 1 | FP_target = 1)
    Composition of expert-labeled values among FP(target) cases.
    """

    fp = (df[f"{target}_e"] == 0) & (df[f"{target}_{suffix}"] == 1)
    n_fp = int(fp.sum())

    if n_fp < min_fp:
        return pd.DataFrame()

    out = []
    for u in values_list:
        if u == target:
            continue

        k = int(((df.loc[fp, f"{u}_e"] == 1)).sum())
        p = k / n_fp if n_fp > 0 else np.nan

        out.append({
            "true_value": u,
            "k": k,                 # absolute count
            "n": n_fp,           # denominator  #n_FP
            "p": float(p),  #P(true=1 | FP_target)
        })

    return (
        pd.DataFrame(out)
        .sort_values("p", ascending=False)
        .reset_index(drop=True)
    )



def fp_copredicted_values(df, target, suffix, values_list, min_fp=10):
    """
    Inside FP(target) posts: what other values does the model also predict?
    Returns P(pred_u=1 | FP_target=1) for each u.
    """
    fp = fp_mask(df, target, suffix)
    n_fp = int(fp.sum())
    if n_fp < min_fp:
        return pd.DataFrame({"pred_value": [], "P(pred=1 | FP_target)": [], "n_fp": []})

    out = []
    for u in values_list:
#         p = (df.loc[fp, f"{u}_{suffix}"] == 1).mean()
        k = int((df.loc[fp, f"{u}_{suffix}"] == 1).sum())
        p = k / n_fp if n_fp else np.nan
        
        out.append({
            "pred_value": u,
            "k": k,
            "n": n_fp,  #n_fp
            "p": float(p),  #P(pred=1 | FP_target)
            
        })

    res = pd.DataFrame(out).sort_values("p", ascending=False).reset_index(drop=True)
    return res

def fn_expert_composition(
    df,
    target,
    suffix,
    values_list,
    min_fn=20
):
    """
    P(true_u = 1 | FN_target = 1)
    Composition of expert-labeled values among FN(target) cases.
    """

    # FN mask: expert=1, model=0
    fn = (df[f"{target}_e"] == 1) & (df[f"{target}_{suffix}"] == 0)
    n_fn = int(fn.sum())

    if n_fn < min_fn:
        return pd.DataFrame()

    out = []
    for u in values_list:
        if u == target:
            continue

#         p = (df.loc[fn, f"{u}_e"] == 1).mean()
        k = int((df.loc[fn, f"{u}_e"] == 1).sum())
        p = k / n_fn if n_fn else np.nan

        out.append({
            "true_value": u,
            "k": k,
            "n": n_fn,  #n_FN
            "p": float(p)  #P(true=1 | FN_target)
            
        })

    return (
        pd.DataFrame(out)
        .sort_values("p", ascending=False)
        .reset_index(drop=True)
    )


def fn_copredicted_values(df, target, suffix, values_list, min_fn=10):
    fn = fn_mask(df, target, suffix)
    n_fn = fn.sum()
    if n_fn < min_fn:
        return pd.DataFrame()

    out = []
    for u in values_list:
#         p = (df.loc[fn, f"{u}_{suffix}"] == 1).mean()
        k = int((df.loc[fn, f"{u}_{suffix}"] == 1).sum())
        p = k / n_fn if n_fn else np.nan
    
        out.append({
            "pred_value": u,
            "k": k,
            "n": int(n_fn),  #n_fn
            "p": float(p)  #P(pred=1 | FN_target)
            
        })

    return pd.DataFrame(out).sort_values("p", ascending=False)


In [68]:
def over_attribution_report(
    df,
    suffix,
    values_list,
    targets,
    mode="composition",   # "composition" or "trigger"
    min_support=30,
    min_fp=20
):
    """
    Returns:
      - summary: FP rates per target
      - triggers: dict[target] -> trigger table (true values that increase FP(target))
      - copred: dict[target] -> co-predicted values within FP(target)
    """
    summary_rows = []
    expert_ctx = {}
    copred = {}

    for t in targets:
        summary_rows.append({
            "target": t,
            "model": suffix,
            "fp_rate": round(fp_rate(df, t, suffix), 4),
            "n_fp": int(fp_mask(df, t, suffix).sum()),
            "n_expert_neg": int((df[f"{t}_e"] == 0).sum())
        })

        if mode == "composition":
            expert_ctx[t] = fp_expert_composition(
                df, target=t, suffix=suffix, values_list=values_list, min_fp=min_fp
            )
        elif mode == "trigger":
            expert_ctx[t] = fp_trigger_by_true_values(
                df, target=t, suffix=suffix, values_list=values_list, min_support=min_support
            )
        else:
            raise ValueError("mode must be 'composition' or 'trigger'")

        copred[t] = fp_copredicted_values(
            df, target=t, suffix=suffix, values_list=values_list
        )

    summary = pd.DataFrame(summary_rows)
    return summary, expert_ctx, copred


Over-attribution report!

In [69]:
     # или "gemini_nr", "gpt5", ...
targets = ('Self-direction','Stimulation','Hedonism','Achievement','Power','Security','Conformity','Tradition','Benevolence','Universalism')

summary_extra, expert_ctx, copred_extra = over_attribution_report(
    df,
    suffix,
    values_list,
    targets, "composition",
    min_support=30,
    min_fp=20
)

summary_extra

,target,model,fp_rate,n_fp,n_expert_neg
0,Self-direction,gemini_ext3,0.0765,68,889
1,Stimulation,gemini_ext3,0.0441,41,930
2,Hedonism,gemini_ext3,0.0379,30,791
3,Achievement,gemini_ext3,0.0389,34,873
4,Power,gemini_ext3,0.0331,32,966
5,Security,gemini_ext3,0.0695,52,748
6,Conformity,gemini_ext3,0.0127,12,945
7,Tradition,gemini_ext3,0.0458,43,939
8,Benevolence,gemini_ext3,0.0654,31,474
9,Universalism,gemini_ext3,0.0545,50,917


Omission report !

In [70]:
def omission_report(df, suffix, values_list, targets, min_support=30):
    exp_comp = {}
    copred = {}
    summary_rows = []
    
    for t in targets:
        
        summary_rows.append({
            "target": t,
            "model": suffix,
            "fn_rate": round(fn_rate(df, t, suffix), 4),
            "n_fn": int(fn_mask(df, t, suffix).sum()),
            "n_expert_pos": int((df[f"{t}_e"] == 1).sum())
        })
        
        
        exp_comp[t] = fn_expert_composition(df, t, suffix, values_list, min_support)
        copred[t] = fn_copredicted_values(df, t, suffix, values_list)
    summary = pd.DataFrame(summary_rows)
    return summary, exp_comp, copred


In [71]:
summary_missed, exp_comp_missed, copred_missed = omission_report(
    df=df,
    suffix=suffix,
    values_list=values_list,
    targets=targets,
    min_support=30
)
summary_missed

,target,model,fn_rate,n_fn,n_expert_pos
0,Self-direction,gemini_ext3,0.2973,33,111
1,Stimulation,gemini_ext3,0.4571,32,70
2,Hedonism,gemini_ext3,0.5120,107,209
3,Achievement,gemini_ext3,0.2992,38,127
4,Power,gemini_ext3,0.2941,10,34
5,Security,gemini_ext3,0.5000,126,252
6,Conformity,gemini_ext3,0.7091,39,55
7,Tradition,gemini_ext3,0.1475,9,61
8,Benevolence,gemini_ext3,0.1730,91,526
9,Universalism,gemini_ext3,0.1928,16,83


In [74]:
from scipy.stats import spearmanr
def vote_count(series: pd.Series) -> pd.Series:
    return (
        series
        .fillna("")
        .astype(str)
        .str.split(",")
        .apply(lambda xs: sum(x.strip() == "1" for x in xs if x != ""))
    )

def compute_spearman_per_value(df, values_list, suffix):
    rows = []

    for v in values_list:
        llm = vote_count(df[f"{v}_{suffix}_raw"])   # 0–5
        exp = vote_count(df[f"{v}_e_raw"])          # 0–3

        mask = ~np.isnan(llm) & ~np.isnan(exp)

        rho, p = spearmanr(exp[mask], llm[mask])

        rows.append({
            "value": v,
            "rho": round(rho, 3),
            "p_value": p
        })

    return pd.DataFrame(rows)

In [75]:
df_corr=compute_spearman_per_value(df, values_list, "gemini_ext3")
df_corr

,value,rho,p_value
0,Self-direction,0.634,1.905216e-113
1,Stimulation,0.535,4.661192e-75
2,Hedonism,0.598,7.264712e-98
3,Achievement,0.681,4.378410e-137
4,Power,0.516,5.106297e-69
5,Security,0.597,2.073123e-97
6,Conformity,0.395,1.130181e-38
7,Tradition,0.584,1.996209e-92
8,Benevolence,0.833,2.628733e-258
9,Universalism,0.674,2.366029e-133


# Second benchmark

In [ ]:
#open second expert benchmark
df_test

In [41]:
df_combined_list=[]
k=3


out = {"precision": [], "recall": [], "f1": []}
all_y_true = []
all_y_pred = []

for i, v in enumerate (values_list):
    df_list[i]=df_list[i].rename(columns={"Do these posts express the value of "+v+"?":"text"})
    dfv=pd.merge(df_list[i], df_test, on='text', how="left")
    
    dfv[f"{v}_e"]=np.where((dfv['expert1'].astype(int)+dfv['expert2'].astype(int)+dfv['expert3'].astype(int))>=2, 1, 0)

#     print (v)
#     print (dfv['experts_majority'].value_counts(normalize=True))
#     print ("")

    df_combined_list.append(dfv)
    y_true = dfv[f'{v}_e'].values
    y_pred = dfv[f"{v}_{suffix}"].values
    all_y_true.append(y_true)
    all_y_pred.append(y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    
    out["precision"].append(round(p,3))
    out["recall"].append(round(r,3))
    out["f1"].append(round(f1,3))
        
        #!!! Add here missed and extra rates deltas !!!


    metrics= {v: np.array(vs, dtype=float) for v, vs in out.items()}
all_y_true = np.concatenate(all_y_true)
all_y_pred = np.concatenate(all_y_pred)

p_micro, r_micro, f1_micro, _ = precision_recall_fscore_support(
    all_y_true,
    all_y_pred,
    average="binary",
    zero_division=0
)

print(round(p_micro, 3), round(r_micro, 3), round(f1_micro, 3))

0.521 0.838 0.642


In [51]:
df_metrics = pd.DataFrame(
    metrics,
    index=values_list
)
print (df_metrics.precision.mean())
print (df_metrics.recall.mean())
print (df_metrics.f1.mean())

df_metrics


0.5178999999999999
0.8058000000000002
0.6113999999999999


,precision,recall,f1
Self-direction,0.472,0.829,0.602
Stimulation,0.538,0.560,0.549
Hedonism,0.453,0.857,0.593
Achievement,0.642,0.872,0.739
Power,0.488,0.800,0.606
Security,0.386,0.944,0.548
Conformity,0.562,0.409,0.474
Tradition,0.706,0.923,0.800
Benevolence,0.643,0.935,0.762
Universalism,0.289,0.929,0.441


In [43]:
from scipy.stats import spearmanr
import pandas as pd
import numpy as np

def compute_spearman_per_value_validation(
    df_combined_list,
    values_list,
    suffix
):
    rows = []

    for i, v in enumerate(values_list):

        dfv = df_combined_list[i]

        # эксперты: 0-3
        exp = (
            dfv["expert1"].astype(int)
            + dfv["expert2"].astype(int)
            + dfv["expert3"].astype(int)
        )

        # Gemini/GPT: 0-5
        llm = (
            dfv[f"{v}_{suffix}_raw"]
            .astype(str)
            .str.split(",")
            .apply(lambda x: sum(int(i) for i in x))
        )

        mask = ~pd.isna(exp) & ~pd.isna(llm)

        rho, p = spearmanr(
            exp[mask],
            llm[mask]
        )

        rows.append({
            "value": v,
            "rho": round(rho, 3),
            "p_value": p
        })

    return pd.DataFrame(rows)

In [44]:
# suffix="gemini_nr3RU"
spearman_val = compute_spearman_per_value_validation(
    df_combined_list,
    values_list,
    suffix
)

spearman_val

,value,rho,p_value
0,Self-direction,0.635,4.768706e-24
1,Stimulation,0.505,2.410892e-14
2,Hedonism,0.742,3.170427e-36
3,Achievement,0.714,1.329331e-32
4,Power,0.581,1.992309e-19
5,Security,0.681,1.303288e-28
6,Conformity,0.445,4.074542e-11
7,Tradition,0.791,3.668948e-44
8,Benevolence,0.719,1.503491e-33
9,Universalism,0.537,1.653282e-16


In [45]:
def error_rates_table(
    df_combined_list,
    values_list,
    suffix
):

    rows = []

    for i, v in enumerate(values_list):

        dfv = df_combined_list[i]

        y_true = dfv[f"{v}_e"]
        y_pred = dfv[f"{v}_{suffix}"]

        fp = ((y_true == 0) & (y_pred == 1)).sum()
        fn = ((y_true == 1) & (y_pred == 0)).sum()

        n_neg = (y_true == 0).sum()
        n_pos = (y_true == 1).sum()

        fp_rate = fp / n_neg if n_neg else np.nan
        fn_rate = fn / n_pos if n_pos else np.nan

        rows.append({
            "value": v,

            "expert_pos": int(n_pos),
            "expert_neg": int(n_neg),

            "FP": int(fp),
            "FN": int(fn),

            "FP_rate": round(fp_rate, 3),
            "FN_rate": round(fn_rate, 3)
        })

    return pd.DataFrame(rows)

In [46]:

# OA and UA rates

error_table = error_rates_table(
    df_combined_list,
    values_list,
    suffix
)

error_table[["value", "FP_rate", "FN_rate"]]

,value,FP_rate,FN_rate
0,Self-direction,0.238,0.171
1,Stimulation,0.069,0.440
2,Hedonism,0.169,0.143
3,Achievement,0.117,0.128
4,Power,0.120,0.200
5,Security,0.329,0.056
6,Conformity,0.039,0.591
7,Tradition,0.093,0.077
8,Benevolence,0.317,0.065
9,Universalism,0.170,0.071
